In [ ]:
# ????
from pathlib import Path
import numpy as np

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

DAT = ROOT / "data" / "79" / "step_003433.dat"
JACOBI = ROOT / "data" / "79" / "jacobi_csr.bin"
OUT = ROOT / "results" / "79"
OUT.mkdir(parents=True, exist_ok=True)

with open(DAT, "r") as f:
    title = f.readline().strip()
    var_line = f.readline().strip()
    data = np.loadtxt(f)

names = [s.strip().strip('"') for s in var_line.split("=", 1)[1].split(",")]
x, y, rho, u, v, T, p, Ma, miubl = (
    data[:, i].astype(np.float64) for i in range(data.shape[1])
)
n = rho.size

print(f"{DAT}  ({DAT.stat().st_size / 1e6:.1f} MB)")
print(title)
print("variables:", names)
print(f"cells = {n},  x in [{x.min():.4f}, {x.max():.4f}],  y in [{y.min():.4f}, {y.max():.4f}]")
print(
    f"rho in [{rho.min():.6f}, {rho.max():.6f}]   "
    f"u in [{u.min():.3f}, {u.max():.3f}]   "
    f"v in [{v.min():.3f}, {v.max():.3f}]   "
    f"T in [{T.min():.3f}, {T.max():.3f}]   "
    f"miubl (file nu_tilde) in [{miubl.min():.3e}, {miubl.max():.3e}]"
)
print(f"Jacobi CSR: {JACOBI}  ({JACOBI.stat().st_size / 1e6:.1f} MB)")


In [ ]:
# ?? W ?????? W ? Jacobi
R_gas = float(np.median(p / (rho * T)))
fast = Ma > 0.1
gamma = float(np.median((u[fast] ** 2 + v[fast] ** 2) / (Ma[fast] ** 2 * p[fast] / rho[fast])))
Cv = R_gas / (gamma - 1.0)

W = np.zeros((n, 5, 5), dtype=np.float64)
W[:, 0, 0] = 1.0
for row in (1, 2, 4):
    W[:, row, row] = 1.0 / rho
W[:, 1, 0] = -u / rho
W[:, 2, 0] = -v / rho
W[:, 3, 0] = -T / rho + (u ** 2 + v ** 2) / (2.0 * Cv * rho)
W[:, 3, 1] = -u / (Cv * rho)
W[:, 3, 2] = -v / (Cv * rho)
W[:, 3, 3] = 1.0 / (Cv * rho)
W[:, 4, 0] = -miubl / rho

ROW = ("rho", "u", "v", "T", "miubl")
COL = ("rho", "rho*u", "rho*v", "rho*E", "rho*miubl")
np.set_printoptions(precision=6, suppress=False, linewidth=110)


def show(i, tag=""):
    print(f"[cell {i}]{(' ' + tag) if tag else ''}  x={x[i]:+.6f} y={y[i]:+.6f}")
    print(
        f"    rho={rho[i]:.6f}  u={u[i]:+.4f}  v={v[i]:+.4f}  "
        f"T={T[i]:.4f}  miubl={miubl[i]:.4e}  Ma={Ma[i]:.4f}"
    )
    s = np.array2string(W[i], formatter={"float_kind": lambda z: f"{z: .6e}"})
    for name, line in zip(ROW, s.splitlines()):
        print(f"    {name:>9} |{line[1:]}")
    print()


print(f"R = {R_gas:.6f} J/(kg.K)")
print(f"gamma = {gamma:.6f}")
print(f"Cv = {Cv:.6f} J/(kg.K)")
print("W.shape =", W.shape, " dtype =", W.dtype, f" memory = {W.nbytes / 1e6:.2f} MB")
print("columns:", ", ".join(COL))
for i in range(3):
    show(i)
for i in np.linspace(0, n - 1, 6).astype(int)[2:]:
    show(int(i))
show(int(np.argmax(Ma)), "max Ma")
show(int(np.argmin(rho)), "min rho")
show(int(np.argmax(miubl)), "max miubl")

w_names = [f"w_{r}_{c}" for r in ROW for c in COL]
np.save(OUT / "W_inv.npy", W)
full = np.column_stack([np.arange(n), x, y, W.reshape(n, 25)])
header = ", ".join(["cell", "x", "y"] + w_names)
np.savetxt(OUT / "W_inv.csv", full, fmt="%.10e", delimiter=",", header=header, comments="")
np.savez(
    OUT / "W_inv_meta.npz",
    cell=np.arange(n), x=x, y=y,
    R_gas=R_gas, gamma=gamma, Cv=Cv,
    fifth_variable="miubl (file field: nu_tilde)",
)

# jacobi_csr.bin?2 ? int64 ????row_ptr?col_ind?4x4 float64 ????
with JACOBI.open("rb") as f:
    n_block_rows, nnz = np.fromfile(f, dtype=np.int64, count=2)
    row_ptr = np.fromfile(f, dtype=np.int64, count=n_block_rows + 1)
    col_ind = np.fromfile(f, dtype=np.int64, count=nnz)
    jacobi_blocks = np.fromfile(f, dtype=np.float64, count=nnz * 16).reshape(nnz, 4, 4)

# The file stores a 4-variable block Jacobian, so only the compatible
# 4x4 principal block of each 5x5 W_i can be applied.
W4 = W[:, :4, :4]
block_rows = np.repeat(np.arange(n_block_rows), np.diff(row_ptr))
wj_blocks = np.einsum("bij,bjk->bik", W4[block_rows], jacobi_blocks)

wj_path = OUT / "WJ_csr.bin"
with wj_path.open("wb") as f:
    np.asarray([n_block_rows, nnz], dtype=np.int64).tofile(f)
    row_ptr.tofile(f)
    col_ind.tofile(f)
    np.asarray(wj_blocks, dtype=np.float64).tofile(f)

print("W4.shape =", W4.shape)
print("Jacobi blocks =", jacobi_blocks.shape)
print("WJ blocks =", wj_blocks.shape)
print(f"{wj_path}  ({wj_path.stat().st_size / 1e6:.1f} MB)")
for filename in ("W_inv.npy", "W_inv.csv", "W_inv_meta.npz"):
    path = OUT / filename
    print(f"{path}  {path.stat().st_size / 1e6:.2f} MB")


In [ ]:
# ??? WJ ??????????? St?0.06 ????????
from scipy.sparse import bsr_matrix, eye
from scipy.sparse.linalg import eigs

N_BLOCK = n
with (OUT / "WJ_csr.bin").open("rb") as f:
    n_block_rows, nnz = np.fromfile(f, dtype=np.int64, count=2)
    row_ptr = np.fromfile(f, dtype=np.int64, count=n_block_rows + 1)
    col_ind = np.fromfile(f, dtype=np.int64, count=nnz)
    wj_blocks = np.fromfile(f, dtype=np.float64, count=nnz * 16).reshape(nnz, 4, 4)

A = bsr_matrix(
    (wj_blocks, col_ind, row_ptr),
    shape=(4 * n_block_rows, 4 * n_block_rows),
    blocksize=(4, 4),
)

stored_scalar_entries = A.data.size
actual_scalar_nnz = np.count_nonzero(A.data)
print(
    f"sparse BSR matrix: shape={A.shape}, block nnz={nnz}, "
    f"stored scalar slots={stored_scalar_entries}, "
    f"actual scalar nnz={actual_scalar_nnz}, "
    f"stored={A.data.nbytes / 1e6 + A.indices.nbytes / 1e6 + A.indptr.nbytes / 1e6:.1f} MB"
)

# ?????????????? 80 ?????????????????? 1 m?
far = np.argsort(np.hypot(x, y))[-80:]
U_inf = float(np.hypot(u[far], v[far]).mean())
chord = 1.0
st_target = 0.06
st_band = (0.04, 0.08)
omega_target = 2.0 * np.pi * st_target * U_inf / chord

# lambda = sigma + i*omega??? sigma ??????????????
eigenvalues, eigenvectors = eigs(
    A,
    k=24,
    sigma=1j * omega_target,
    which="LM",
    tol=1e-8,
    maxiter=5000,
    ncv=80,
)

st_values = eigenvalues.imag * chord / (2.0 * np.pi * U_inf)
positive = eigenvalues.imag > 0.0
in_band = positive & (st_values >= st_band[0]) & (st_values <= st_band[1])
if not np.any(in_band):
    raise RuntimeError(f"???? St ? {st_band} ????????")

candidate_indices = np.flatnonzero(in_band)
selected_index = candidate_indices[np.argmax(eigenvalues.real[candidate_indices])]
selected = eigenvalues[selected_index]
selected_st = st_values[selected_index]

order = candidate_indices[np.argsort(eigenvalues.real[candidate_indices])[::-1]]
print(f"A.shape = {A.shape}, nnz = {A.nnz}")
print(f"U_inf = {U_inf:.9f} m/s, chord = {chord:.9f} m")
print(f"target: St = {st_target:.6f}, omega = {omega_target:.9f} rad/s")
print("candidates in St band (sorted by growth rate):")
for i in order:
    z = eigenvalues[i]
    print(f"  lambda = {z.real:+.12e} {z.imag:+.12e}j, St = {st_values[i]:.9f}")
print(
    f"selected: lambda = {selected.real:+.12e} {selected.imag:+.12e}j, "
    f"St = {selected_st:.9f}, growth rate = {selected.real:+.12e}"
)

np.savez(
    OUT / "eigenvalues_WJ.npz",
    eigenvalues=eigenvalues,
    eigenvectors=eigenvectors,
    U_inf=U_inf,
    chord=chord,
    target_St=st_target,
    target_omega=omega_target,
    band=np.asarray(st_band),
    selected_index=selected_index,
)
np.savetxt(
    OUT / "eigenvalues_WJ.csv",
    np.column_stack([eigenvalues.real, eigenvalues.imag, st_values]),
    delimiter=",",
    header="growth_rate,angular_frequency,St",
    comments="",
)
